# 01. EDA: LoL Challenger Ranked Games

목표는 10분/15분 시점의 경기 데이터를 이용해 `blueWins`를 예측하는 것입니다.
이 노트북에서는 데이터 크기, 결측치, 타깃 분포, 주요 차이 피처와 승패의 관계를 확인합니다.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import summarize_dataset, prepare_xy
from src.feature_engineering import engineer_features

DATA_10 = PROJECT_ROOT / "data" / "Challenger_Ranked_Games_10minute.csv"
DATA_15 = PROJECT_ROOT / "data" / "Challenger_Ranked_Games_15minute.csv"

df10 = pd.read_csv(DATA_10)
df15 = pd.read_csv(DATA_15)

print("10minute:", df10.shape)
print("15minute:", df15.shape)

In [ ]:
pd.DataFrame([
    {"time": "10minute", **summarize_dataset(df10)},
    {"time": "15minute", **summarize_dataset(df15)},
])

In [ ]:
print(df10.dtypes.value_counts())
print("\nObject columns:", df10.select_dtypes(include="object").columns.tolist())

df10.head()

## 타깃 분포 확인

`blueWins`가 0이면 블루팀 패배, 1이면 블루팀 승리입니다.
두 클래스 비율이 거의 50:50이면 accuracy도 해석하기 쉬운 편입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, df, title in [(axes[0], df10, "10minute"), (axes[1], df15, "15minute")]:
    df["blueWins"].value_counts().sort_index().plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_xticklabels(["Lose", "Win"], rotation=0)
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## 데이터 누수 주의

`redWins`는 `blueWins`의 정확한 반대 값입니다. 따라서 모델 입력에 넣으면 실제 예측 문제가 아니라 정답을 알려주는 문제가 됩니다. 코드에서는 `gameId`, `redWins`, `blueWins`를 자동으로 제거합니다.

In [ ]:
print("redWins == 1 - blueWins in 10minute:", (df10["redWins"] == 1 - df10["blueWins"]).all())
print("redWins == 1 - blueWins in 15minute:", (df15["redWins"] == 1 - df15["blueWins"]).all())

## 차이 피처 생성

블루팀 절대 수치보다 `블루팀 - 레드팀` 차이가 승패 예측에 더 직접적인 의미를 가집니다.
예를 들어 블루팀 골드가 높아도 레드팀보다 낮으면 불리한 상황입니다.

In [ ]:
eng10 = engineer_features(df10)
eng15 = engineer_features(df15)

important_diff_cols = [
    "diff_totalGolds", "diff_currentGolds", "diff_totalLevel", "diff_kill",
    "diff_assist", "diff_totalMinionKills", "diff_dragon", "diff_towerKills",
    "diff_riftHeralds", "diff_objectiveScore", "diff_KDA"
]

eng10[["blueWins", *important_diff_cols]].head()

In [ ]:
# 승리/패배별 주요 차이 피처 평균
summary = eng10.groupby("blueWins")[important_diff_cols].mean().T
summary.columns = ["blueLose_mean", "blueWin_mean"]
summary["win_minus_lose"] = summary["blueWin_mean"] - summary["blueLose_mean"]
summary.sort_values("win_minus_lose", ascending=False)

In [ ]:
# 주요 피처와 blueWins의 상관계수
corr10 = eng10.select_dtypes("number").corr(numeric_only=True)["blueWins"].sort_values(key=lambda s: s.abs(), ascending=False)
corr10.head(20)

In [ ]:
# 모델 입력으로 실제 사용되는 X 형태 확인
X10, y10 = prepare_xy(df10)
X15, y15 = prepare_xy(df15)
print("X10:", X10.shape)
print("X15:", X15.shape)
print(X10.columns[:20].tolist())